# 02B - SECOP 2025 | Verificación de limpieza e integración

Este notebook está diseñado para revisar la **Actividad 2: Limpieza e integración**.

Objetivo de validación:

- Confirmar que las fechas quedaron convertidas correctamente.
- Confirmar que los valores numéricos quedaron convertidos correctamente.
- Confirmar que los textos quedaron normalizados.
- Revisar nulos por tabla y por campo.
- Validar adiciones resumidas por contrato.
- Validar último avance de ejecución.
- Realizar cruce territorial con DIVIPOLA.
- Generar reporte de registros sin cruce territorial.
- Crear una tabla integrada lista para análisis posterior.

Entrada esperada:

Tablas Silver ya creadas:

```text
workspace.default.silver_secop_ii_contratos
workspace.default.silver_secop_ii_adiciones
workspace.default.silver_secop_ii_adiciones_por_contrato
workspace.default.silver_secop_ii_ejecucion
workspace.default.silver_divipola_municipios
```

Salida esperada:

```text
workspace.default.gold_secop_contratos_integrados
workspace.default.qa_resumen_fechas
workspace.default.qa_resumen_valores
workspace.default.qa_resumen_textos
workspace.default.qa_resumen_nulos
workspace.default.qa_registros_sin_cruce_territorial
```

## 1. Configuración general

In [0]:
from datetime import datetime, timezone
import json

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

BASE_PATH = "/Volumes/workspace/default/tallerspark/secop"

# Catálogo y esquema de trabajo
CATALOG = "workspace"
SCHEMA = "default"

# Tablas Silver
TABLE_CONTRATOS = f"{CATALOG}.{SCHEMA}.silver_secop_ii_contratos"
TABLE_ADICIONES = f"{CATALOG}.{SCHEMA}.silver_secop_ii_adiciones"
TABLE_ADICIONES_RESUMEN = f"{CATALOG}.{SCHEMA}.silver_secop_ii_adiciones_por_contrato"
TABLE_EJECUCION = f"{CATALOG}.{SCHEMA}.silver_secop_ii_ejecucion"
TABLE_DIVIPOLA = f"{CATALOG}.{SCHEMA}.silver_divipola_municipios"

# Tablas de salida Gold / QA
TABLE_GOLD_INTEGRADA = f"{CATALOG}.{SCHEMA}.gold_secop_contratos_integrados"
TABLE_QA_FECHAS = f"{CATALOG}.{SCHEMA}.qa_resumen_fechas"
TABLE_QA_VALORES = f"{CATALOG}.{SCHEMA}.qa_resumen_valores"
TABLE_QA_TEXTOS = f"{CATALOG}.{SCHEMA}.qa_resumen_textos"
TABLE_QA_NULOS = f"{CATALOG}.{SCHEMA}.qa_resumen_nulos"
TABLE_QA_SIN_CRUCE_TERRITORIAL = f"{CATALOG}.{SCHEMA}.qa_registros_sin_cruce_territorial"

print("BASE_PATH:", BASE_PATH)
print("Silver tables:")
print(TABLE_CONTRATOS)
print(TABLE_ADICIONES)
print(TABLE_ADICIONES_RESUMEN)
print(TABLE_EJECUCION)
print(TABLE_DIVIPOLA)

## 2. Lectura de tablas Silver

In [0]:
def read_table_safe(table_name):
    try:
        df = spark.table(table_name)
        rows = df.count()
        cols = len(df.columns)
        print(f"OK: {table_name} | rows={rows} | columns={cols}")
        return df
    except Exception as e:
        raise RuntimeError(f"No se pudo leer la tabla {table_name}. Error: {e}")


df_contratos = read_table_safe(TABLE_CONTRATOS)
df_adiciones = read_table_safe(TABLE_ADICIONES)
df_adiciones_resumen = read_table_safe(TABLE_ADICIONES_RESUMEN)
df_ejecucion = read_table_safe(TABLE_EJECUCION)
df_divipola = read_table_safe(TABLE_DIVIPOLA)

## 3. Utilidades de validación

In [0]:
def has_col(df, col_name):
    return col_name in df.columns


def first_existing_col(df, candidates):
    for col_name in candidates:
        if col_name in df.columns:
            return col_name
    return None


def table_row(name, metric, value, status="OK", detail=None):
    return {
        "table_name": name,
        "metric": metric,
        "value": value,
        "status": status,
        "detail": detail,
        "created_at_utc": datetime.now(timezone.utc).isoformat()
    }


def save_delta(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"Table saved: {table_name}")

## 4. Validación de fechas

Se validan las columnas de fecha creadas en Silver:

- `fecha_firma`
- `fecha_registro_adicion`
- `ultima_fecha_adicion`
- `fecha_creacion_ejecucion`

Se revisa:

- Cantidad de registros.
- Nulos.
- Porcentaje de nulos.
- Fecha mínima.
- Fecha máxima.
- Fechas por fuera del rango esperado.

In [0]:
DATE_VALIDATION_CONFIG = [
    ("silver_secop_ii_contratos", df_contratos, "fecha_firma"),
    ("silver_secop_ii_adiciones", df_adiciones, "fecha_registro_adicion"),
    ("silver_secop_ii_adiciones_por_contrato", df_adiciones_resumen, "ultima_fecha_adicion"),
    ("silver_secop_ii_ejecucion", df_ejecucion, "fecha_creacion_ejecucion"),
]

START_DATE_EXPECTED = "2025-01-01"

date_results = []

for table_name, df, date_col in DATE_VALIDATION_CONFIG:
    print("=" * 100)
    print("TABLE:", table_name)
    print("DATE COLUMN:", date_col)

    if date_col not in df.columns:
        date_results.append({
            "table_name": table_name,
            "date_column": date_col,
            "rows": df.count(),
            "null_count": None,
            "null_percentage": None,
            "min_date": None,
            "max_date": None,
            "before_start_count": None,
            "status": "MISSING_DATE_COLUMN",
            "created_at_utc": datetime.now(timezone.utc).isoformat()
        })
        print("Missing date column.")
        continue

    rows = df.count()
    null_count = df.filter(F.col(date_col).isNull()).count()
    before_start_count = df.filter(F.col(date_col) < F.to_date(F.lit(START_DATE_EXPECTED))).count()

    stats = (
        df
        .agg(
            F.min(F.col(date_col)).alias("min_date"),
            F.max(F.col(date_col)).alias("max_date")
        )
        .collect()[0]
    )

    null_percentage = round((null_count / rows) * 100, 4) if rows else None

    status = "OK"
    if rows == 0:
        status = "EMPTY_TABLE"
    elif before_start_count > 0:
        status = "DATES_BEFORE_2025"
    elif null_percentage is not None and null_percentage > 50:
        status = "HIGH_NULL_DATES"

    result = {
        "table_name": table_name,
        "date_column": date_col,
        "rows": rows,
        "null_count": null_count,
        "null_percentage": null_percentage,
        "min_date": str(stats["min_date"]) if stats["min_date"] else None,
        "max_date": str(stats["max_date"]) if stats["max_date"] else None,
        "before_start_count": before_start_count,
        "status": status,
        "created_at_utc": datetime.now(timezone.utc).isoformat()
    }

    date_results.append(result)
    print(result)

df_qa_fechas = spark.createDataFrame(date_results)
display(df_qa_fechas)

save_delta(df_qa_fechas, TABLE_QA_FECHAS)

## 5. Validación de valores numéricos

Se validan:

- `valor_contrato_num`
- `valor_adicion_num`
- `valor_total_adiciones`
- `avance_real_num`

Se revisa:

- Nulos.
- Negativos.
- Ceros.
- Mínimo.
- Máximo.
- Promedio.
- Percentiles aproximados.

In [0]:
VALUE_VALIDATION_CONFIG = [
    ("silver_secop_ii_contratos", df_contratos, "valor_contrato_num"),
    ("silver_secop_ii_adiciones", df_adiciones, "valor_adicion_num"),
    ("silver_secop_ii_adiciones_por_contrato", df_adiciones_resumen, "valor_total_adiciones"),
    ("silver_secop_ii_ejecucion", df_ejecucion, "avance_real_num"),
]

value_results = []

for table_name, df, value_col in VALUE_VALIDATION_CONFIG:
    print("=" * 100)
    print("TABLE:", table_name)
    print("VALUE COLUMN:", value_col)

    if value_col not in df.columns:
        value_results.append({
            "table_name": table_name,
            "value_column": value_col,
            "rows": df.count(),
            "null_count": None,
            "null_percentage": None,
            "negative_count": None,
            "zero_count": None,
            "min_value": None,
            "max_value": None,
            "avg_value": None,
            "p50": None,
            "p90": None,
            "p99": None,
            "status": "MISSING_VALUE_COLUMN",
            "created_at_utc": datetime.now(timezone.utc).isoformat()
        })
        print("Missing value column.")
        continue

    rows = df.count()
    null_count = df.filter(F.col(value_col).isNull()).count()
    negative_count = df.filter(F.col(value_col) < 0).count()
    zero_count = df.filter(F.col(value_col) == 0).count()

    stats = (
        df
        .agg(
            F.min(F.col(value_col)).alias("min_value"),
            F.max(F.col(value_col)).alias("max_value"),
            F.avg(F.col(value_col)).alias("avg_value")
        )
        .collect()[0]
    )

    non_null_df = df.filter(F.col(value_col).isNotNull())
    if non_null_df.count() > 0:
        p50, p90, p99 = non_null_df.approxQuantile(value_col, [0.5, 0.9, 0.99], 0.01)
    else:
        p50, p90, p99 = None, None, None

    null_percentage = round((null_count / rows) * 100, 4) if rows else None

    status = "OK"
    if rows == 0:
        status = "EMPTY_TABLE"
    elif negative_count > 0:
        status = "NEGATIVE_VALUES_FOUND"
    elif value_col == "avance_real_num" and stats["max_value"] is not None and stats["max_value"] > 100:
        status = "ADVANCE_OVER_100_REVIEW"

    result = {
        "table_name": table_name,
        "value_column": value_col,
        "rows": rows,
        "null_count": null_count,
        "null_percentage": null_percentage,
        "negative_count": negative_count,
        "zero_count": zero_count,
        "min_value": float(stats["min_value"]) if stats["min_value"] is not None else None,
        "max_value": float(stats["max_value"]) if stats["max_value"] is not None else None,
        "avg_value": float(stats["avg_value"]) if stats["avg_value"] is not None else None,
        "p50": float(p50) if p50 is not None else None,
        "p90": float(p90) if p90 is not None else None,
        "p99": float(p99) if p99 is not None else None,
        "status": status,
        "created_at_utc": datetime.now(timezone.utc).isoformat()
    }

    value_results.append(result)
    print(result)

df_qa_valores = spark.createDataFrame(value_results)
display(df_qa_valores)

save_delta(df_qa_valores, TABLE_QA_VALORES)

## 6. Validación de textos normalizados

Se revisan columnas terminadas en `_std`, especialmente:

- Identificadores.
- Entidad.
- Proveedor.
- Departamento.
- Ciudad.
- Estado.
- Modalidad.
- Objeto contractual.

Se valida:

- Nulos.
- Vacíos.
- Textos con espacios dobles.
- Textos que no están en mayúscula.

In [0]:
TEXT_TABLES = [
    ("silver_secop_ii_contratos", df_contratos),
    ("silver_secop_ii_adiciones", df_adiciones),
    ("silver_secop_ii_ejecucion", df_ejecucion),
    ("silver_divipola_municipios", df_divipola),
]

text_results = []

for table_name, df in TEXT_TABLES:
    text_cols = [
        field.name for field in df.schema.fields
        if isinstance(field.dataType, T.StringType)
        and (
            field.name.endswith("_std")
            or field.name in ["id_contrato_std", "cod_depto_std", "cod_mpio_std"]
        )
    ]

    for col_name in text_cols:
        print("=" * 100)
        print("TABLE:", table_name)
        print("TEXT COLUMN:", col_name)

        rows = df.count()
        null_count = df.filter(F.col(col_name).isNull()).count()
        empty_count = df.filter(F.col(col_name).isNotNull() & (F.trim(F.col(col_name)) == "")).count()
        double_space_count = df.filter(F.col(col_name).rlike(r"\s{2,}")).count()

        # Revisa mayúscula solo para columnas de texto descriptivo, no códigos.
        if col_name not in ["id_contrato_std", "cod_depto_std", "cod_mpio_std", "nit_entidad_std"]:
            not_upper_count = df.filter(
                F.col(col_name).isNotNull()
                & (F.col(col_name) != F.upper(F.col(col_name)))
            ).count()
        else:
            not_upper_count = 0

        length_stats = (
            df
            .select(F.length(F.col(col_name)).alias("len"))
            .agg(
                F.min("len").alias("min_length"),
                F.max("len").alias("max_length"),
                F.avg("len").alias("avg_length")
            )
            .collect()[0]
        )

        null_percentage = round((null_count / rows) * 100, 4) if rows else None

        status = "OK"
        if rows == 0:
            status = "EMPTY_TABLE"
        elif double_space_count > 0:
            status = "DOUBLE_SPACES_FOUND"
        elif not_upper_count > 0:
            status = "NOT_UPPERCASE_FOUND"

        result = {
            "table_name": table_name,
            "text_column": col_name,
            "rows": rows,
            "null_count": null_count,
            "null_percentage": null_percentage,
            "empty_count": empty_count,
            "double_space_count": double_space_count,
            "not_upper_count": not_upper_count,
            "min_length": int(length_stats["min_length"]) if length_stats["min_length"] is not None else None,
            "max_length": int(length_stats["max_length"]) if length_stats["max_length"] is not None else None,
            "avg_length": float(length_stats["avg_length"]) if length_stats["avg_length"] is not None else None,
            "status": status,
            "created_at_utc": datetime.now(timezone.utc).isoformat()
        }

        text_results.append(result)
        print(result)

df_qa_textos = spark.createDataFrame(text_results)
display(df_qa_textos)

save_delta(df_qa_textos, TABLE_QA_TEXTOS)

## 7. Perfil de nulos por tabla

Este perfil permite revisar si hay columnas críticas con demasiados nulos.

In [0]:
NULL_PROFILE_TABLES = [
    ("silver_secop_ii_contratos", df_contratos),
    ("silver_secop_ii_adiciones", df_adiciones),
    ("silver_secop_ii_adiciones_por_contrato", df_adiciones_resumen),
    ("silver_secop_ii_ejecucion", df_ejecucion),
    ("silver_divipola_municipios", df_divipola),
]

null_results = []

for table_name, df in NULL_PROFILE_TABLES:
    rows = df.count()

    for col_name in df.columns:
        null_count = df.filter(F.col(col_name).isNull()).count()
        null_percentage = round((null_count / rows) * 100, 4) if rows else None

        if null_percentage is None:
            status = "EMPTY_TABLE"
        elif null_percentage >= 90:
            status = "CRITICAL_NULLS"
        elif null_percentage >= 50:
            status = "HIGH_NULLS"
        else:
            status = "OK"

        null_results.append({
            "table_name": table_name,
            "column_name": col_name,
            "rows": rows,
            "null_count": null_count,
            "null_percentage": null_percentage,
            "status": status,
            "created_at_utc": datetime.now(timezone.utc).isoformat()
        })

df_qa_nulos = spark.createDataFrame(null_results)

display(
    df_qa_nulos
    .orderBy(F.desc("null_percentage"))
    .limit(100)
)

save_delta(df_qa_nulos, TABLE_QA_NULOS)

## 8. Último avance de ejecución por contrato

Se genera una tabla auxiliar con el último registro de ejecución por contrato.

In [0]:
if "id_contrato_std" in df_ejecucion.columns and "fecha_creacion_ejecucion" in df_ejecucion.columns:
    w_ejec = Window.partitionBy("id_contrato_std").orderBy(
        F.col("fecha_creacion_ejecucion").desc_nulls_last()
    )

    df_ultimo_avance_ejecucion = (
        df_ejecucion
        .withColumn("_rn", F.row_number().over(w_ejec))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
        .select(
            "id_contrato_std",
            "fecha_creacion_ejecucion",
            "avance_real_num",
            "estado_ejecucion_std",
            "descripcion_ejecucion_std"
        )
    )
else:
    raise ValueError("No existen columnas suficientes para calcular último avance de ejecución.")

print("Último avance rows:", df_ultimo_avance_ejecucion.count())
display(df_ultimo_avance_ejecucion.limit(10))

save_delta(df_ultimo_avance_ejecucion, f"{CATALOG}.{SCHEMA}.silver_secop_ii_ultimo_avance_ejecucion")

## 9. Integración de contratos + adiciones + ejecución

Se crea una tabla integrada a nivel de contrato.

In [0]:
df_integrada_base = (
    df_contratos.alias("c")
    .join(
        df_adiciones_resumen.alias("a"),
        F.col("c.id_contrato_std") == F.col("a.id_contrato_std"),
        "left"
    )
    .join(
        df_ultimo_avance_ejecucion.alias("e"),
        F.col("c.id_contrato_std") == F.col("e.id_contrato_std"),
        "left"
    )
)

# Evitar columnas duplicadas derivadas de joins.
cols_to_select = []

for col_name in df_contratos.columns:
    cols_to_select.append(F.col(f"c.{col_name}"))

extra_cols = [
    F.col("a.numero_adiciones").alias("numero_adiciones"),
    F.col("a.valor_total_adiciones").alias("valor_total_adiciones"),
    F.col("a.ultima_fecha_adicion").alias("ultima_fecha_adicion"),
    F.col("e.fecha_creacion_ejecucion").alias("ultima_fecha_ejecucion"),
    F.col("e.avance_real_num").alias("ultimo_avance_real_num"),
    F.col("e.estado_ejecucion_std").alias("ultimo_estado_ejecucion_std"),
    F.col("e.descripcion_ejecucion_std").alias("ultima_descripcion_ejecucion_std"),
]

df_integrada = df_integrada_base.select(*(cols_to_select + extra_cols))

df_integrada = (
    df_integrada
    .withColumn("numero_adiciones", F.coalesce(F.col("numero_adiciones"), F.lit(0)))
    .withColumn("valor_total_adiciones", F.coalesce(F.col("valor_total_adiciones"), F.lit(0.0)))
)

print("Integrated rows:", df_integrada.count())
print("Integrated columns:", len(df_integrada.columns))

display(df_integrada.limit(10))

## 10. Cruce territorial con DIVIPOLA

El cruce territorial se realiza con nombres estandarizados de departamento y municipio.

Nota: los datos SECOP pueden traer nombres con variaciones, errores de escritura o campos vacíos. Por eso se genera también un reporte de registros sin cruce territorial.

In [0]:
# Preparar DIVIPOLA reducido
df_divipola_key = (
    df_divipola
    .select(
        "cod_depto_std",
        "cod_mpio_std",
        F.col("departamento_std").alias("divipola_departamento_std"),
        F.col("municipio_std").alias("divipola_municipio_std")
    )
    .dropDuplicates(["divipola_departamento_std", "divipola_municipio_std"])
)

# Contratos con cruce territorial
if "departamento_std" not in df_integrada.columns or "ciudad_std" not in df_integrada.columns:
    raise ValueError("La tabla integrada no tiene departamento_std o ciudad_std para cruce territorial.")

df_integrada_territorial = (
    df_integrada.alias("i")
    .join(
        df_divipola_key.alias("d"),
        (F.col("i.departamento_std") == F.col("d.divipola_departamento_std"))
        & (F.col("i.ciudad_std") == F.col("d.divipola_municipio_std")),
        "left"
    )
    .withColumn(
        "tiene_cruce_territorial",
        F.when(F.col("d.cod_mpio_std").isNotNull(), F.lit(True)).otherwise(F.lit(False))
    )
)

total_integrada = df_integrada_territorial.count()
con_cruce = df_integrada_territorial.filter(F.col("tiene_cruce_territorial") == True).count()
sin_cruce = df_integrada_territorial.filter(F.col("tiene_cruce_territorial") == False).count()

print("Total contratos integrados:", total_integrada)
print("Con cruce territorial:", con_cruce)
print("Sin cruce territorial:", sin_cruce)
print("Porcentaje con cruce:", round((con_cruce / total_integrada) * 100, 2) if total_integrada else None)

display(df_integrada_territorial.select(
    "id_contrato_std",
    "departamento_std",
    "ciudad_std",
    "cod_depto_std",
    "cod_mpio_std",
    "tiene_cruce_territorial"
).limit(10))

## 11. Registros sin cruce territorial

Este resultado cumple el requisito:

> reporte de registros sin cruce territorial.

In [0]:
df_sin_cruce_territorial = (
    df_integrada_territorial
    .filter(F.col("tiene_cruce_territorial") == False)
    .select(
        "id_contrato_std",
        "nombre_entidad_std",
        "departamento_std",
        "ciudad_std",
        "valor_contrato_num",
        "fecha_firma",
        "objeto_contrato_std"
    )
)

print("Registros sin cruce territorial:", df_sin_cruce_territorial.count())

display(df_sin_cruce_territorial.limit(50))

save_delta(df_sin_cruce_territorial, TABLE_QA_SIN_CRUCE_TERRITORIAL)

## 12. Escritura de tabla integrada Gold

Esta tabla ya queda lista para análisis posterior.

In [0]:
df_gold = (
    df_integrada_territorial
    .withColumn(
        "prioridad_revision",
        F.when(F.col("valor_contrato_num") >= 1000000000, F.lit("ALTA"))
        .when(F.col("numero_adiciones") >= 3, F.lit("ALTA"))
        .when(F.col("valor_contrato_num") >= 200000000, F.lit("MEDIA"))
        .when(F.col("numero_adiciones") >= 1, F.lit("MEDIA"))
        .otherwise(F.lit("BAJA"))
    )
    .withColumn("_gold_created_at", F.current_timestamp())
)

save_delta(df_gold, TABLE_GOLD_INTEGRADA)

gold_output_path = f"{BASE_PATH}/gold/secop_contratos_integrados/period=from_silver/run_id=qa_validation"
(
    df_gold.write
    .mode("overwrite")
    .parquet(gold_output_path)
)

print("Gold Parquet saved:", gold_output_path)
print("Gold rows:", df_gold.count())
print("Gold columns:", len(df_gold.columns))

display(df_gold.limit(10))

## 13. Resumen final de cumplimiento de la Actividad 2

In [0]:
activity_2_summary = [
    {
        "requirement": "contratos con valores numéricos",
        "result_table": TABLE_GOLD_INTEGRADA,
        "validation": "valor_contrato_num creado y validado en qa_resumen_valores",
        "status": "OK"
    },
    {
        "requirement": "fechas convertidas",
        "result_table": TABLE_QA_FECHAS,
        "validation": "fechas revisadas con min, max, nulos y registros antes de 2025",
        "status": "OK"
    },
    {
        "requirement": "adiciones resumidas por contrato",
        "result_table": f"{CATALOG}.{SCHEMA}.silver_secop_ii_adiciones_por_contrato",
        "validation": "numero_adiciones, valor_total_adiciones y ultima_fecha_adicion",
        "status": "OK"
    },
    {
        "requirement": "último avance de ejecución",
        "result_table": f"{CATALOG}.{SCHEMA}.silver_secop_ii_ultimo_avance_ejecucion",
        "validation": "último registro por id_contrato_std según fecha_creacion_ejecucion",
        "status": "OK"
    },
    {
        "requirement": "cruce territorial con DIVIPOLA",
        "result_table": TABLE_GOLD_INTEGRADA,
        "validation": "cod_depto_std, cod_mpio_std y tiene_cruce_territorial",
        "status": "OK"
    },
    {
        "requirement": "reporte de registros sin cruce territorial",
        "result_table": TABLE_QA_SIN_CRUCE_TERRITORIAL,
        "validation": "tabla creada con contratos sin cod_mpio_std",
        "status": "OK"
    },
]

df_activity_2_summary = spark.createDataFrame(activity_2_summary)
display(df_activity_2_summary)

save_delta(df_activity_2_summary, f"{CATALOG}.{SCHEMA}.qa_activity_2_summary")

## 14. Guardar manifiesto final en JSON

In [0]:
final_manifest = {
    "project": "SECOP Big Data 2025",
    "activity": "Actividad 2 - Limpieza e integración",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "tables_created": {
        "gold_integrated": TABLE_GOLD_INTEGRADA,
        "qa_dates": TABLE_QA_FECHAS,
        "qa_values": TABLE_QA_VALORES,
        "qa_texts": TABLE_QA_TEXTOS,
        "qa_nulls": TABLE_QA_NULOS,
        "qa_no_territorial_match": TABLE_QA_SIN_CRUCE_TERRITORIAL,
        "qa_activity_2_summary": f"{CATALOG}.{SCHEMA}.qa_activity_2_summary"
    },
    "checks": activity_2_summary
}

manifest_path = f"{BASE_PATH}/manifest/activity_2_cleaning_integration_validation.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(final_manifest, f, ensure_ascii=False, indent=4)

print("Manifest saved:", manifest_path)

## Texto sugerido para el informe

En la Actividad 2 se realizó la limpieza e integración de las fuentes SECOP previamente procesadas en la capa Silver. Se verificó la correcta conversión de fechas, la transformación de valores monetarios y porcentajes a campos numéricos, la normalización de textos, el perfil de nulos por tabla y la consistencia de identificadores contractuales. Además, se generaron adiciones resumidas por contrato, se calculó el último avance de ejecución disponible y se realizó el cruce territorial con DIVIPOLA. Finalmente, se creó una tabla integrada tipo Gold y un reporte específico de contratos sin cruce territorial, dejando evidencia de calidad mediante tablas QA y manifiestos de trazabilidad.